In [0]:
%run /Workspace/Users/admin@jkalexanderhotmail.onmicrosoft.com/ADLSoauth

In [0]:
from pyspark.sql.types import *

# Mapeo CDM → Spark
type_map = {
    "Int64": LongType(),
    "String": StringType(),
    "DateTime": TimestampType(),
    "Decimal": DecimalType(18,2),
    "Double": DoubleType(),
    "Boolean": BooleanType()
}

def build_schema_from_cdm(cdm_path):
    cdm_json = spark.read.option("multiline","true").json(cdm_path).collect()[0].asDict()

    attrs = cdm_json["definitions"][0]["hasAttributes"]

    fields = [
        StructField(a["name"], type_map.get(a["dataFormat"], StringType()), True)
        for a in attrs
    ]

    return StructType(fields)


def read_cdm_entity(cdm_path, csv_path, header=False, delimiter=","):
    schema = build_schema_from_cdm(cdm_path)

    return (spark.read
            .schema(schema)
            .option("header", str(header).lower())
            .option("delimiter", delimiter)
            .csv(csv_path))

In [0]:
base = "abfss://proyecto01@adlsproyecto01.dfs.core.windows.net/ProyectoOAON/Tables/HR"

In [0]:
df = read_cdm_entity(
    f"{base}/WorkerTable.cdm.json",
    f"{base}/WorkerTable/*.csv"
)

display(df)

In [0]:
def read_manifest(manifest_path, base_folder):
    manifest = (spark.read
                .option("multiline","true")
                .json(manifest_path)
                .collect()[0]
                .asDict())

    dfs = {}

    for e in manifest["entities"]:
        name = e["entityName"]

        dfs[name] = read_cdm_entity(
            f"{base_folder}/{name}.cdm.json",
            f"{base_folder}/{name}/*.csv"
        )

    return dfs

In [0]:
tables = read_manifest(
    "abfss://proyecto01@adlsproyecto01.dfs.core.windows.net/ProyectoOAON/Tables/Purchase/Purchase.manifest.cdm.json",
    "abfss://proyecto01@adlsproyecto01.dfs.core.windows.net/ProyectoOAON/Tables/Purchase"
)

display(tables["PurchItem"])